# HRHUB – Final Canonical Implementation

**A Bilateral Job Posting Enriched Semantic Matching System**

This notebook represents the final, canonical implementation of HRHUB.
It consolidates the best-performing logic, metrics, and design decisions
from previous iterations (v3.1, v3.2, v3.2_clean, v4.0).

Purpose:
- Reproduce and validate the results reported in the thesis
- Provide a clear, ordered, and reproducible pipeline
- Serve as the reference for later productionization (Streamlit & HF)

This notebook is research-oriented and follows CRISP-DM principles.

## 0. Scope, Assumptions, and Guarantees

This notebook guarantees:
- Correct reproduction of reported metrics (coverage, fairness, baselines)
- Bilateral matching (candidate → company, company → candidate)
- Explicit separation between:
  - core matching logic
  - evaluation
  - visualization

This notebook does NOT:
- Implement Streamlit UI
- Deploy to Hugging Face
- Optimize for production latency beyond academic requirements

## 📓 SECTION 1 — Data Loading & Integrity Checks

Objective:
- Load all raw datasets used in HRHUB
- Verify dataset sizes match those reported in the thesis
- Perform basic integrity checks before any transformation

This step corresponds to the **Data Understanding** phase of CRISP-DM.
No data modification is performed in this section.

🧱 Section 1.2 — Imports & Configuration

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

🧱 Section 1.3 — Define Data Paths

In [2]:
# Define the data directory 
from pathlib import Path

DATA_DIR = Path("../csv_files")
print("Data directory:", DATA_DIR.resolve())


Data directory: /home/roger/Desktop/hrhub/data/csv_files


🧱 Section 1.4 — Load Core Datasets

In [3]:
# Core datasets
candidates_df = pd.read_csv(DATA_DIR / "resume_data.csv")
companies_df = pd.read_csv(DATA_DIR / "companies.csv")
postings_df = pd.read_csv(DATA_DIR / "postings.csv")

print("Candidates:", candidates_df.shape)
print("Companies:", companies_df.shape)
print("Job postings:", postings_df.shape)

Candidates: (9544, 35)
Companies: (24473, 10)
Job postings: (123849, 31)


🧱 Section 1.5 — Load Supporting Tables

In [4]:
# Supporting tables
skills_df = pd.read_csv(DATA_DIR / "skills.csv")
company_industries_df = pd.read_csv(DATA_DIR / "company_industries.csv")
company_specialties_df = pd.read_csv(DATA_DIR / "company_specialities.csv")
job_skills_df = pd.read_csv(DATA_DIR / "job_skills.csv")

print("Skills:", skills_df.shape)
print("Company industries:", company_industries_df.shape)
print("Company specialties:", company_specialties_df.shape)
print("Job-skill mappings:", job_skills_df.shape)

Skills: (35, 2)
Company industries: (24375, 2)
Company specialties: (169387, 2)
Job-skill mappings: (213768, 2)


🧱 Section 1.6 — Integrity Checks

In [5]:
# Basic integrity checks
assert candidates_df.shape[0] > 0, "Candidates dataset is empty"
assert companies_df.shape[0] > 0, "Companies dataset is empty"
assert postings_df.shape[0] > 0, "Postings dataset is empty"

print("Company columns:", list(companies_df.columns))
print("Candidate columns:", list(candidates_df.columns))

print("Basic integrity checks passed.")

Company columns: ['company_id', 'name', 'description', 'company_size', 'state', 'country', 'city', 'zip_code', 'address', 'url']
Candidate columns: ['address', 'career_objective', 'skills', 'educational_institution_name', 'degree_names', 'passing_years', 'educational_results', 'result_types', 'major_field_of_studies', 'professional_company_names', 'company_urls', 'start_dates', 'end_dates', 'related_skils_in_job', 'positions', 'locations', 'responsibilities', 'extra_curricular_activity_types', 'extra_curricular_organization_names', 'extra_curricular_organization_links', 'role_positions', 'languages', 'proficiency_levels', 'certification_providers', 'certification_skills', 'online_links', 'issue_dates', 'expiry_dates', '\ufeffjob_position_name', 'educationaL_requirements', 'experiencere_requirement', 'age_requirement', 'responsibilities.1', 'skills_required', 'matched_score']
Basic integrity checks passed.


🧱 Section 1.7 — Quick Preview

In [6]:
display(candidates_df.head(3))
display(companies_df.head(3))
display(postings_df.head(3))

,address,career_objective,skills,educational_institution_name,degree_names,passing_years,educational_results,result_types,major_field_of_studies,professional_company_names,company_urls,start_dates,end_dates,related_skils_in_job,positions,locations,responsibilities,extra_curricular_activity_types,extra_curricular_organization_names,extra_curricular_organization_links,role_positions,languages,proficiency_levels,certification_providers,certification_skills,online_links,issue_dates,expiry_dates,﻿job_position_name,educationaL_requirements,experiencere_requirement,age_requirement,responsibilities.1,skills_required,matched_score
0,NaN,Big data analytics working and database wareho...,"['Big Data', 'Hadoop', 'Hive', 'Python', 'Mapr...",['The Amity School of Engineering & Technology...,['B.Tech'],['2019'],['N/A'],[None],['Electronics'],['Coca-COla'],[None],['Nov 2019'],['Till Date'],[['Big Data']],['Big Data Analyst'],['N/A'],Technical Support\nTroubleshooting\nCollaborat...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Senior Software Engineer,B.Sc in Computer Science & Engineering from a ...,At least 1 year,NaN,Technical Support\nTroubleshooting\nCollaborat...,NaN,0.850000
1,NaN,Fresher looking to join as a data analyst and ...,"['Data Analysis', 'Data Analytics', 'Business ...","['Delhi University - Hansraj College', 'Delhi ...","['B.Sc (Maths)', 'M.Sc (Science) (Statistics)']","['2015', '2018']","['N/A', 'N/A']","['N/A', 'N/A']","['Mathematics', 'Statistics']",['BIB Consultancy'],['N/A'],['Sep 2019'],['Till Date'],"[['Data Analysis', 'Business Analysis', 'Machi...",['Business Analyst'],['N/A'],Machine Learning Leadership\nCross-Functional ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Machine Learning (ML) Engineer,M.Sc in Computer Science & Engineering or in a...,At least 5 year(s),NaN,Machine Learning Leadership\nCross-Functional ...,NaN,0.750000
2,NaN,NaN,"['Software Development', 'Machine Learning', '...","['Birla Institute of Technology (BIT), Ranchi']",['B.Tech'],['2018'],['N/A'],['N/A'],['Electronics/Telecommunication'],['Axis Bank Limited'],['N/A'],['June 2018'],['Till Date'],"[['Unified Payment Interface', 'Risk Predictio...",['Software Developer (Machine Learning Enginee...,['N/A'],"Trade Marketing Executive\nBrand Visibility, S...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"Executive/ Senior Executive- Trade Marketing, ...",Master of Business Administration (MBA),At least 3 years,NaN,"Trade Marketing Executive\nBrand Visibility, S...",Brand Promotion\nCampaign Management\nField Su...,0.416667


,company_id,name,description,company_size,state,country,city,zip_code,address,url
0,1009,IBM,"At IBM, we do more than work. We create. We cr...",7.0,NY,US,"Armonk, New York",10504,International Business Machines Corp.,https://www.linkedin.com/company/ibm
1,1016,GE HealthCare,Every day millions of people feel the impact o...,7.0,0,US,Chicago,0,-,https://www.linkedin.com/company/gehealthcare
2,1025,Hewlett Packard Enterprise,Official LinkedIn of Hewlett Packard Enterpris...,7.0,Texas,US,Houston,77389,1701 E Mossy Oaks Rd Spring,https://www.linkedin.com/company/hewlett-packa...


,job_id,company_name,title,description,max_salary,pay_period,location,company_id,views,med_salary,min_salary,formatted_work_type,applies,original_listed_time,remote_allowed,job_posting_url,application_url,application_type,expiry,closed_time,formatted_experience_level,skills_desc,listed_time,posting_domain,sponsored,work_type,currency,compensation_type,normalized_salary,zip_code,fips
0,921716,Corcoran Sawyer Smith,Marketing Coordinator,Job descriptionA leading real estate firm in N...,20.0,HOURLY,"Princeton, NJ",2774458.0,20.0,NaN,17.0,Full-time,2.0,1.713398e+12,NaN,https://www.linkedin.com/jobs/view/921716/?trk...,NaN,ComplexOnsiteApply,1.715990e+12,NaN,NaN,Requirements: \n\nWe are seeking a College or ...,1.713398e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,38480.0,8540.0,34021.0
1,1829192,NaN,Mental Health Therapist/Counselor,"At Aspen Therapy and Wellness , we are committ...",50.0,HOURLY,"Fort Collins, CO",NaN,1.0,NaN,30.0,Full-time,NaN,1.712858e+12,NaN,https://www.linkedin.com/jobs/view/1829192/?tr...,NaN,ComplexOnsiteApply,1.715450e+12,NaN,NaN,NaN,1.712858e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,83200.0,80521.0,8069.0
2,10998357,The National Exemplar,Assitant Restaurant Manager,The National Exemplar is accepting application...,65000.0,YEARLY,"Cincinnati, OH",64896719.0,8.0,NaN,45000.0,Full-time,NaN,1.713278e+12,NaN,https://www.linkedin.com/jobs/view/10998357/?t...,NaN,ComplexOnsiteApply,1.715870e+12,NaN,NaN,We are currently accepting resumes for FOH - A...,1.713278e+12,NaN,0,FULL_TIME,USD,BASE_SALARY,55000.0,45202.0,39061.0


## 🧠 SECTION 2 — Vocabulary Normalization & Skill Expansion

Objective:
- Normalize column names and textual fields
- Resolve abbreviated and inconsistent skill representations
- Map skill identifiers to full textual descriptions
- Prepare clean, semantically meaningful inputs for semantic models (SBERT)

Motivation:
LLMs and embedding models perform better on semantically complete text.

This step corresponds to the **Data Preparation** phase of CRISP-DM.
No matching or modeling is performed in this section.

In [7]:
# Section 2.1 — Column name normalization

def normalize_columns(df):
    df = df.copy()
    df.columns = (
        df.columns
        .str.replace("\ufeff", "", regex=False)  # remove BOM
        .str.strip()
        .str.lower()
    )
    return df

candidates_df = normalize_columns(candidates_df)
companies_df  = normalize_columns(companies_df)
postings_df   = normalize_columns(postings_df)

print("Candidate columns (normalized):", list(candidates_df.columns))
print("Company columns (normalized):", list(companies_df.columns))
print("Posting columns (normalized):", list(postings_df.columns))

Candidate columns (normalized): ['address', 'career_objective', 'skills', 'educational_institution_name', 'degree_names', 'passing_years', 'educational_results', 'result_types', 'major_field_of_studies', 'professional_company_names', 'company_urls', 'start_dates', 'end_dates', 'related_skils_in_job', 'positions', 'locations', 'responsibilities', 'extra_curricular_activity_types', 'extra_curricular_organization_names', 'extra_curricular_organization_links', 'role_positions', 'languages', 'proficiency_levels', 'certification_providers', 'certification_skills', 'online_links', 'issue_dates', 'expiry_dates', 'job_position_name', 'educational_requirements', 'experiencere_requirement', 'age_requirement', 'responsibilities.1', 'skills_required', 'matched_score']
Company columns (normalized): ['company_id', 'name', 'description', 'company_size', 'state', 'country', 'city', 'zip_code', 'address', 'url']
Posting columns (normalized): ['job_id', 'company_name', 'title', 'description', 'max_salary

### Section 2.2 — Candidate Field Selection

Some columns in the resume dataset contain downstream or job-related
information that must not be used for representation learning or matching.

In this section, we explicitly define which candidate fields are allowed
to contribute to the semantic representation.

In [8]:
# Section 2.2 — Candidate allowed fields

CANDIDATE_ALLOWED_FIELDS = [
    "career_objective",
    "skills",
    "degree_names",
    "major_field_of_studies",
    "positions",
    "responsibilities",
    "languages",
    "certification_skills"
]

missing = [c for c in CANDIDATE_ALLOWED_FIELDS if c not in candidates_df.columns]
print("Missing allowed fields:", missing)

print("Allowed candidate fields:", CANDIDATE_ALLOWED_FIELDS)


Missing allowed fields: []
Allowed candidate fields: ['career_objective', 'skills', 'degree_names', 'major_field_of_studies', 'positions', 'responsibilities', 'languages', 'certification_skills']


### Section 2.3 — Skill Vocabulary Reference

The skills table provides a mapping from skill identifiers or abbreviations
to full textual descriptions.

This table will be used later during company enrichment and text construction.

In [9]:
# Section 2.3 — Load and inspect skills vocabulary

skills_df = normalize_columns(skills_df)

display(skills_df.head())
print("Total skills in vocabulary:", skills_df.shape[0])


,skill_abr,skill_name
0,ART,Art/Creative
1,DSGN,Design
2,ADVR,Advertising
3,PRDM,Product Management
4,DIST,Distribution


Total skills in vocabulary: 35


Section 2.4 — Null & Empty Value Sanity Check

In [10]:
# Section 2.4 — Null value inspection

candidates_df[CANDIDATE_ALLOWED_FIELDS].isnull().mean().sort_values(ascending=False)

languages                 0.926655
certification_skills      0.789606
career_objective          0.503353
degree_names              0.008801
major_field_of_studies    0.008801
positions                 0.008801
skills                    0.005868
responsibilities          0.000000
dtype: float64

## SECTION 3 — Company Enrichment via Job Posting Bridge

Objective:
- Enrich company profiles using information extracted from job postings
- Explicitly associate companies with required skills and job titles
- Increase semantic coverage for companies with sparse descriptions

This section implements the **core HRHUB contribution**:
job postings act as a semantic bridge between candidates and companies.

This step still belongs to the **Data Preparation** phase of CRISP-DM.

In [11]:
# Section 3.1 — Job postings linked to companies

print("Posting columns:", list(postings_df.columns))

postings_df[["job_id", "company_id", "title"]].head()

Posting columns: ['job_id', 'company_name', 'title', 'description', 'max_salary', 'pay_period', 'location', 'company_id', 'views', 'med_salary', 'min_salary', 'formatted_work_type', 'applies', 'original_listed_time', 'remote_allowed', 'job_posting_url', 'application_url', 'application_type', 'expiry', 'closed_time', 'formatted_experience_level', 'skills_desc', 'listed_time', 'posting_domain', 'sponsored', 'work_type', 'currency', 'compensation_type', 'normalized_salary', 'zip_code', 'fips']


,job_id,company_id,title
0,921716,2774458.0,Marketing Coordinator
1,1829192,NaN,Mental Health Therapist/Counselor
2,10998357,64896719.0,Assitant Restaurant Manager
3,23221523,766262.0,Senior Elder Law / Trusts and Estates Associat...
4,35982263,NaN,Service Technician


In [12]:
# Section 3.2 — Filter postings with valid company_id

valid_postings_df = postings_df.dropna(subset=["company_id"]).copy()
valid_postings_df["company_id"] = valid_postings_df["company_id"].astype(int)

print("Total postings:", postings_df.shape[0])
print("Valid postings:", valid_postings_df.shape[0])

Total postings: 123849
Valid postings: 122132


This number is what later gives the ~96% coverage.

In [13]:
# Section 3.3 — Aggregate job titles per company

company_job_titles = (
    valid_postings_df
    .groupby("company_id")["title"]
    .apply(lambda x: list(x.dropna().unique()))
    .reset_index(name="job_titles")
)

company_job_titles.head()

,company_id,job_titles
0,1009,"[Business Sales & Delivery Executive - SAP, Pr..."
1,1016,"[VP of Engineering, Demand Planning Leader - M..."
2,1025,"[Federal IT Call Center Technician (TS/SCI, Fu..."
3,1028,"[Associate, Corporate Development, Customer Su..."
4,1033,[Workday Certified Project Manager – Midwest M...


In [14]:
# Section 3.4 — Load job-skill mappings

job_skills_df = normalize_columns(job_skills_df)

print(job_skills_df.head())
print("Total job-skill relations:", job_skills_df.shape[0])

       job_id skill_abr
0  3884428798      MRKT
1  3884428798        PR
2  3884428798       WRT
3  3887473071      SALE
4  3887465684       FIN
Total job-skill relations: 213768


In [15]:
# Section 3.5 — Merge job skills with postings to reach companies

job_company_skills = (
    valid_postings_df[["job_id", "company_id"]]
    .merge(job_skills_df, on="job_id", how="inner")
)

print(job_company_skills.head())
print("Company-skill rows:", job_company_skills.shape[0])

     job_id  company_id skill_abr
0    921716     2774458      MRKT
1    921716     2774458      SALE
2  10998357    64896719      MGMT
3  10998357    64896719      MNFC
4  23221523      766262      OTHR
Company-skill rows: 203168


In [16]:
# Section 3.6 — Aggregate skills per company (using skill_abr)

company_skills = (
    job_company_skills
    .groupby("company_id")["skill_abr"]
    .apply(lambda x: list(x.unique()))
    .reset_index(name="skill_abrs")
)

company_skills.head()

,company_id,skill_abrs
0,1009,"[IT, PRDM, OTHR, SALE, FIN, DSGN, BD, MRKT]"
1,1016,"[ENG, IT, OTHR, PRJM, DSGN, ART, HR, EDU, TRNG..."
2,1025,"[IT, PRJM, SALE, BD, PRDM, MRKT, ENG, QA]"
3,1028,"[BD, SALE, OTHR, RSCH, ANLS, IT, FIN, MGMT, EN..."
4,1033,"[STRA, IT, CNSL, SALE, ENG, MGMT, MNFC, PRJM, ..."


In [17]:
# Section 3.7 — Build enriched company table

companies_enriched = companies_df.merge(
    company_job_titles, on="company_id", how="left"
).merge(
    company_skills, on="company_id", how="left"
)

companies_enriched.head()

,company_id,name,description,company_size,state,country,city,zip_code,address,url,job_titles,skill_abrs
0,1009,IBM,"At IBM, we do more than work. We create. We cr...",7.0,NY,US,"Armonk, New York",10504,International Business Machines Corp.,https://www.linkedin.com/company/ibm,"[Business Sales & Delivery Executive - SAP, Pr...","[IT, PRDM, OTHR, SALE, FIN, DSGN, BD, MRKT]"
1,1016,GE HealthCare,Every day millions of people feel the impact o...,7.0,0,US,Chicago,0,-,https://www.linkedin.com/company/gehealthcare,"[VP of Engineering, Demand Planning Leader - M...","[ENG, IT, OTHR, PRJM, DSGN, ART, HR, EDU, TRNG..."
2,1025,Hewlett Packard Enterprise,Official LinkedIn of Hewlett Packard Enterpris...,7.0,Texas,US,Houston,77389,1701 E Mossy Oaks Rd Spring,https://www.linkedin.com/company/hewlett-packa...,"[Federal IT Call Center Technician (TS/SCI, Fu...","[IT, PRJM, SALE, BD, PRDM, MRKT, ENG, QA]"
3,1028,Oracle,We’re a cloud technology company that provides...,7.0,Texas,US,Austin,78741,2300 Oracle Way,https://www.linkedin.com/company/oracle,"[Associate, Corporate Development, Customer Su...","[BD, SALE, OTHR, RSCH, ANLS, IT, FIN, MGMT, EN..."
4,1033,Accenture,Accenture is a leading global professional ser...,7.0,0,IE,Dublin 2,0,Grand Canal Harbour,https://www.linkedin.com/company/accenture,[Workday Certified Project Manager – Midwest M...,"[STRA, IT, CNSL, SALE, ENG, MGMT, MNFC, PRJM, ..."


In [18]:
# Section 3.8 — Enrichment coverage

coverage = companies_enriched["skill_abrs"].notna().mean()

print(f"Company skill enrichment coverage: {coverage:.4f}")

Company skill enrichment coverage: 0.9614


## 4. Aligned Text Construction

Objective:
- Convert structured data into aligned natural language templates
- Ensure candidates and companies share a comparable semantic space

Includes:
- CandidateTextBuilder
- CompanyTextBuilder

## SECTION 4 — Aligned Text Construction

Objective:
- Build aligned natural language representations for candidates and companies
- Ensure missing values do not propagate into embeddings
- Produce semantically comparable text inputs for representation learning

This section converts structured data into text.
No embeddings or matching are performed here.

In [19]:
# Section 4.1 — Safe text utilities

def safe_str(x):
    if pd.isna(x):
        return ""
    return str(x).strip()

def join_non_empty(parts, sep=" | "):
    parts = [safe_str(p) for p in parts if safe_str(p)]
    return sep.join(parts)

### Section 4.2 — Candidate Text Representation

Candidate text is built from a selected subset of resume fields.
Fields with missing values are skipped.
No job-related or downstream fields are included.

In [20]:
# Section 4.2 — Build candidate text

def build_candidate_text(row):
    return join_non_empty([
        row.get("career_objective"),
        row.get("skills"),
        row.get("degree_names"),
        row.get("major_field_of_studies"),
        row.get("positions"),
        row.get("responsibilities"),
        row.get("certification_skills"),
        row.get("languages"),
    ])

candidates_df["candidate_text"] = candidates_df.apply(build_candidate_text, axis=1)

candidates_df[["candidate_text"]].head()

,candidate_text
0,Big data analytics working and database wareho...
1,Fresher looking to join as a data analyst and ...
2,"['Software Development', 'Machine Learning', '..."
3,To obtain a position in a fast-paced business ...
4,Professional accountant with an outstanding wo...


Section 4.3 — Expand Company Skill Abbreviations

Now we convert skill abbreviations → full descriptions

In [21]:
# Section 4.3 — Skill abbreviation to full description mapping

# assume skills_df has columns like: skill_abr, skill_name (adjust if needed)
skill_map = dict(
    zip(skills_df["skill_abr"], skills_df.iloc[:, 1])
)

def expand_skill_abrs(skill_abrs):
    if not isinstance(skill_abrs, list):
        return []
    return [skill_map.get(abr, abr) for abr in skill_abrs]

companies_enriched["expanded_skills"] = companies_enriched["skill_abrs"].apply(expand_skill_abrs)

companies_enriched[["expanded_skills"]].head()

,expanded_skills
0,"[Information Technology, Product Management, O..."
1,"[Engineering, Information Technology, Other, P..."
2,"[Information Technology, Project Management, S..."
3,"[Business Development, Sales, Other, Research,..."
4,"[Strategy/Planning, Information Technology, Co..."


### Section 4.4 — Company Text Representation

Company text combines:
- static company description
- enriched job titles
- enriched skill descriptions

This ensures companies and candidates share a comparable semantic space.

In [22]:
# Section 4.4 — Build company text

def build_company_text(row):
    return join_non_empty([
        row.get("description"),
        " ".join(row.get("job_titles", [])) if isinstance(row.get("job_titles"), list) else "",
        " ".join(row.get("expanded_skills", [])) if isinstance(row.get("expanded_skills"), list) else "",
    ])

companies_enriched["company_text"] = companies_enriched.apply(build_company_text, axis=1)

companies_enriched[["company_text"]].head()

,company_text
0,"At IBM, we do more than work. We create. We cr..."
1,Every day millions of people feel the impact o...
2,Official LinkedIn of Hewlett Packard Enterpris...
3,We’re a cloud technology company that provides...
4,Accenture is a leading global professional ser...


In [23]:
# Section 4.5 — Sanity checks

print("Empty candidate texts:", (candidates_df["candidate_text"] == "").mean())
print("Empty company texts:", (companies_enriched["company_text"] == "").mean())

Empty candidate texts: 0.0
Empty company texts: 0.0


## 5. Semantic Embeddings

Objective:
- Generate dense vector representations for candidates and companies
- Use a shared embedding space (all-MiniLM-L6-v2)

Model characteristics:
- 6-layer Transformer
- 22M parameters
- 384-dimensional embeddings

## SECTION 5 — Semantic Embeddings

Objective:
- Generate dense vector representations for candidates and companies
- Use a shared semantic space based on SBERT (MiniLM)
- Persist embeddings for reuse in matching and evaluation

This step corresponds to the **Modeling** phase of CRISP-DM.

In [24]:
# Section 5.1 — Load sentence transformer model
from sentence_transformers import SentenceTransformer
import torch

model_name = "all-MiniLM-L6-v2"
model = SentenceTransformer(model_name, device="cpu")

print("Loaded model on device:", model.device)


/home/roger/Desktop/hrhub/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded model on device: cpu


In [33]:
# Section 5.2 — Candidate embeddings

candidate_texts = candidates_df["candidate_text"].tolist()

candidate_embeddings = model.encode(
    candidate_texts,
    show_progress_bar=True,
    convert_to_numpy=True
)

candidate_embeddings.shape

Batches: 100%|██████████| 299/299 [04:47<00:00,  1.04it/s]


(9544, 384)

In [34]:
# Section 5.3 — Company embeddings

company_texts = companies_enriched["company_text"].tolist()

company_embeddings = model.encode(
    company_texts,
    show_progress_bar=True,
    convert_to_numpy=True
)

company_embeddings.shape

Batches: 100%|██████████| 765/765 [12:24<00:00,  1.03it/s]


(24473, 384)

In [35]:
# Section 5.4 — Save Embeddings (FINAL FOLDER)
import numpy as np
import json

np.save(EMB_DIR / "candidate_embeddings.npy", candidate_embeddings)
np.save(EMB_DIR / "company_embeddings.npy", company_embeddings)

metadata = {
    "model": model_name,
    "candidate_count": candidate_embeddings.shape[0],
    "company_count": company_embeddings.shape[0],
    "embedding_dim": candidate_embeddings.shape[1]
}

with open(FINAL_DIR / "model_info.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("Embeddings and metadata saved to:", FINAL_DIR)


Embeddings and metadata saved to: ../final


In [ ]:
# Section 5.5 — Quick Sanity Check
np.linalg.norm(candidate_embeddings[0]), np.linalg.norm(company_embeddings[0])

## 6. Bilateral Similarity Matching

Objective:
- Compute cosine similarity between candidates and companies
- Support both:
  - Candidate → Company
  - Company → Candidate

Outputs:
- Similarity matrix
- Top-K matches for both perspectives

## SECTION 6 — Bilateral Similarity Matching

Objective:
- Compute semantic similarity between candidates and companies
- Support both matching directions:
  - Candidate → Company
  - Company → Candidate

This section produces similarity scores only.
No evaluation metrics are computed here.

In [25]:
# Section 6.1 — Load Embeddings (FINAL folder)
import numpy as np
from pathlib import Path

FINAL_DIR = Path("../final")
EMB_DIR = FINAL_DIR / "embeddings"

candidate_embeddings = np.load(EMB_DIR / "candidate_embeddings.npy")
company_embeddings   = np.load(EMB_DIR / "company_embeddings.npy")

candidate_embeddings.shape, company_embeddings.shape

((9544, 384), (24473, 384))

In [26]:
# Section 6.2 — Cosine Similarity Matrix
from sklearn.metrics.pairwise import cosine_similarity

# Candidate → Company similarity
similarity_matrix = cosine_similarity(candidate_embeddings, company_embeddings)

similarity_matrix.shape

(9544, 24473)

In [27]:
# Section 6.3 — Candidate → Company Top-K

TOP_K = 10

candidate_topk = np.argsort(-similarity_matrix, axis=1)[:, :TOP_K]

candidate_topk[:3]

array([[19209,  6545, 20497,  8494, 15301,  6383, 21965, 21236,  7065,
        21370],
       [ 4917, 21571,   387,  6502,     0, 21391, 19529, 21044, 19209,
        21965],
       [16602, 18062, 23593,   172,  9421, 21391, 18456, 20938, 24009,
        23994]])

In [28]:
# Section 6.4 — Company → Candidate Top-K

company_topk = np.argsort(-similarity_matrix.T, axis=1)[:, :TOP_K]

company_topk[:3]


array([[3752, 7782,   63, 6863, 6068,  176, 4892, 8161, 6882, 1416],
       [2901, 1827, 9403, 7072, 4430, 7381, 5383, 2853, 1873, 8623],
       [ 176, 7685, 4694, 2710, 4622, 4592, 9355,  217, 4504, 5592]])

In [29]:
# Section 6.5 — Wrap Matches with Scores
def get_candidate_matches(candidate_idx, topk=TOP_K):
    idxs = candidate_topk[candidate_idx][:topk]
    scores = similarity_matrix[candidate_idx, idxs]
    return list(zip(idxs, scores))

def get_company_matches(company_idx, topk=TOP_K):
    idxs = company_topk[company_idx][:topk]
    scores = similarity_matrix.T[company_idx, idxs]
    return list(zip(idxs, scores))

In [30]:
# Section 6.6 Sanity check one candidate and one company

print("Candidate 0 → Companies:", get_candidate_matches(0))
print("Company 0 → Candidates:", get_company_matches(0))


Candidate 0 → Companies: [(19209, 0.7015388), (6545, 0.69418764), (20497, 0.68982136), (8494, 0.6641093), (15301, 0.65597296), (6383, 0.6514117), (21965, 0.64888084), (21236, 0.6485373), (7065, 0.6471101), (21370, 0.64525646)]
Company 0 → Candidates: [(3752, 0.7173019), (7782, 0.71677667), (63, 0.7154684), (6863, 0.71434027), (6068, 0.71139836), (176, 0.71122545), (4892, 0.7111286), (8161, 0.7109983), (6882, 0.7099373), (1416, 0.7096757)]


## 7. Baseline Models

Objective:
- Provide academic baselines for comparison

Baselines:
- TF-IDF + Cosine Similarity
- Jaccard / Keyword Overlap

Note:
Baselines are not tuned and exist solely for validation purposes.

## SECTION 7 — Baseline Models (TF-IDF & Jaccard)

Objective:
- Establish non-neural baselines for semantic matching
- Compare SBERT against classical text-based methods
- Support academic validation claims

Baselines are not tuned and are used solely for comparison.

In [31]:
# Section 7.1 — Shared corpora

candidate_texts = candidates_df["candidate_text"].tolist()
company_texts   = companies_enriched["company_text"].tolist()

len(candidate_texts), len(company_texts)

(9544, 24473)

In [32]:
# Section 7.2 — TF-IDF embeddings

from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1, 2),
    stop_words="english"
)

tfidf_candidate = tfidf.fit_transform(candidate_texts)
tfidf_company   = tfidf.transform(company_texts)

tfidf_candidate.shape, tfidf_company.shape


((9544, 19625), (24473, 19625))

In [ ]:
# Section 7.3 — TF-IDF cosine similarity

from sklearn.metrics.pairwise import cosine_similarity

tfidf_similarity = cosine_similarity(tfidf_candidate, tfidf_company)
tfidf_similarity.shape

In [ ]:
# Section 7.4 — TF-IDF top-K

TOP_K = 10

tfidf_candidate_topk = np.argsort(-tfidf_similarity, axis=1)[:, :TOP_K]
tfidf_company_topk   = np.argsort(-tfidf_similarity.T, axis=1)[:, :TOP_K]

tfidf_candidate_topk[:3]

array([[ 3048,  6545, 11541, 21370,  8494,  7496, 16951, 18053, 20497,
         5981],
       [15756, 20433, 16717, 16635, 18525,  2910,  1706, 14478, 22943,
        13601],
       [22625, 10925,  9062, 21960, 18600, 15656, 15005, 15884, 18677,
        10209]])

In [ ]:
# Section 7.5 — Jaccard similarity utilities

def jaccard(a, b):
    a = set(a.split())
    b = set(b.split())
    if not a or not b:
        return 0.0
    return len(a & b) / len(a | b)


In [ ]:
# Section 7.6 — Jaccard similarity (sampled)

import random

SAMPLE_CANDIDATES = 500
SAMPLE_COMPANIES  = 500

cand_idx = random.sample(range(len(candidate_texts)), SAMPLE_CANDIDATES)
comp_idx = random.sample(range(len(company_texts)), SAMPLE_COMPANIES)

jaccard_sim = np.zeros((SAMPLE_CANDIDATES, SAMPLE_COMPANIES))

for i, ci in enumerate(cand_idx):
    for j, cj in enumerate(comp_idx):
        jaccard_sim[i, j] = jaccard(candidate_texts[ci], company_texts[cj])

jaccard_sim.shape


(500, 500)

In [ ]:
# Section 7.7 — Sanity check

print("TF-IDF similarity range:",
      tfidf_similarity.min(), tfidf_similarity.max())

print("Jaccard similarity range:",
      jaccard_sim.min(), jaccard_sim.max())


TF-IDF similarity range: 0.0 0.501653049487335
Jaccard similarity range: 0.002304147465437788 0.1092436974789916


## 8. Evaluation Metrics

Metrics:
- Match score distribution
- Bilateral fairness ratio
- Coverage
- Embedding quality statistics

Goal:
- Validate claims made in the thesis

## SECTION 8 — Evaluation Metrics

Objective:
- Quantitatively evaluate matching quality
- Compare SBERT against baselines
- Measure coverage, fairness, and embedding behavior

This section reproduces the metrics reported in the thesis.

In [38]:
# Section 8.1 — Match score distribution (sampled for speed)

import numpy as np

np.random.seed(42)

N_CAND = similarity_matrix.shape[0]
N_COMP = similarity_matrix.shape[1]

SAMPLE_CAND = min(500, N_CAND)
TOP_K_EVAL = 10  # for distribution consistency with report

cand_idx = np.random.choice(N_CAND, SAMPLE_CAND, replace=False)

scores = []
for i in cand_idx:
    topk = np.argsort(-similarity_matrix[i])[:TOP_K_EVAL]
    scores.extend(similarity_matrix[i, topk])

scores = np.array(scores)

scores.mean(), np.median(scores), scores.std()


(0.57591504, 0.5779238, 0.047039784)

### Section 8.2 — Bilateral Fairness (Mutual Matches)

Fairness is evaluated on **mutually selected candidate–company pairs**.
Only matches where both sides rank each other within their top-K are considered.

This avoids bias caused by asymmetric population sizes and low-relevance matches,
and ensures fairness is measured on **actual decision outcomes**, not raw score distributions.

In [39]:
# Section 8.2 — Bilateral fairness using rank-based mutual matches

TOP_K_FAIRNESS = 1000
np.random.seed(42)

N_CAND, N_COMP = similarity_matrix.shape

# Precompute rankings
cand_rankings = np.argsort(-similarity_matrix, axis=1)
comp_rankings = np.argsort(-similarity_matrix.T, axis=1)

cand_sample = np.random.choice(N_CAND, min(500, N_CAND), replace=False)

cand_utils = []
comp_utils = []

for c in cand_sample:
    cand_topk = cand_rankings[c, :TOP_K_FAIRNESS]

    for rank_c, comp in enumerate(cand_topk):
        # check mutuality
        comp_topk = comp_rankings[comp, :TOP_K_FAIRNESS]
        if c in comp_topk:
            # normalized rank utility (higher is better)
            cand_util = 1 - (rank_c / TOP_K_FAIRNESS)
            comp_rank = np.where(comp_topk == c)[0][0]
            comp_util = 1 - (comp_rank / TOP_K_FAIRNESS)

            cand_utils.append(cand_util)
            comp_utils.append(comp_util)

cand_utils = np.array(cand_utils)
comp_utils = np.array(comp_utils)

cand_mean = cand_utils.mean()
comp_mean = comp_utils.mean()
fairness_ratio = comp_mean / cand_mean

cand_mean, comp_mean, fairness_ratio


(0.546514741606362, 0.5832030454905771, 1.067131407611033)

In [42]:
# Section 8.3 — Job posting coverage

coverage = companies_enriched["skill_abrs"].notna().mean()
coverage


0.9613860172434928

In [43]:
# Section 8.4 — Embedding quality statistics

cand_norms = np.linalg.norm(candidate_embeddings, axis=1)

cand_norms.mean(), cand_norms.std()


(1.0, 4.590921e-08)

In [44]:
# Section 8.5 — Baseline comparison summary

summary = {
    "SBERT": {
        "mean": float(scores.mean()),
        "std": float(scores.std())
    },
    "TF-IDF": {
        "max": float(tfidf_similarity.max())
    },
    "Jaccard": {
        "max": float(jaccard_sim.max())
    }
}

summary


{'SBERT': {'mean': 0.5759150385856628, 'std': 0.047039784491062164},
 'TF-IDF': {'max': 0.501653049487335},
 'Jaccard': {'max': 0.11864406779661017}}

## 9. Synthetic Validation

Objective:
- Test the system on controlled, synthetic cases
- Validate expected matching behavior

Rationale:
Real-world data lacks explicit ground truth.
Synthetic cases provide controlled validation.

## 10. Visualization and Exploratory Analysis

Includes:
- t-SNE embedding projections
- Bipartite candidate–company graphs
- Skill distribution heatmaps
- Fairness visual diagnostics

Purpose:
- Human interpretability
- Error analysis
- Presentation support

## 11. Discussion, Limitations, and Next Steps

Discussion:
- Strengths of HRHUB
- Comparison with baselines

Limitations:
- Static embeddings
- No online learning
- Dataset biases

Next Steps:
- Modularization into Python packages
- Streamlit integration
- Hugging Face deployment